In [1]:
using DrWatson
@quickactivate "project"
using BlackBoxOptim, Random, Statistics
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

Целевая функция: минимизируем пиковую заболеваемость и смертность

In [2]:
function cost_multi(x)

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/uzarb/work/study/2026-1/2026-1==study-simulation-modeling/labs/lab04/project/notebooks/sir_optimize_parameters/In[2]#1:23\[90mIn[2]:1:23[0;0m]8;;\
function cost_multi(x)[48;2;120;70;70m[0;0m
[90m#                     └ ── [0;0m[91mpremature end of input[0;0m

x[1]: β_und, x[2]: death_rate, x[3]: detection_time

In [3]:
model = initialize_sir(;
Ns = [1000, 1000, 1000],
β_und = fill(x[1], 3),
β_det = fill(x[1]/10, 3),
infection_period = 14,
detection_time = round(Int, x[2]),
death_rate = x[3],
reinfection_probability = 0.1,
Is = [0, 0, 1],
seed = 42,
n_steps = 100,
)
infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
dead_count(model) = 3000 - nagents(model)
peak_infected = 0.0

LoadError: UndefVarError: `x` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Запускаем с несколькими повторами

In [4]:
replicates = 5
peak_vals = Float64[]
dead_vals = Int[]
for rep = 1:replicates
model = initialize_sir(;
Ns = [1000, 1000, 1000],
β_und = fill(x[1], 3),
β_det = fill(x[1]/10, 3),
infection_period = 14,
detection_time = round(Int, x[2]),
death_rate = x[3],
reinfection_probability = 0.1,
Is = [0, 0, 1],
seed = 42 + rep,
n_steps = 100,
)
for step = 1:100
Agents.step!(model, 1)
frac = infected_frac(model)
if frac > peak_infected
peak_infected = frac
end
end
push!(peak_vals, peak_infected)
push!(dead_vals, dead_count(model))
end
return (mean(peak_vals), mean(dead_vals) / 3000) # доля умерших
end

LoadError: UndefVarError: `x` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Запуск оптимизации

In [5]:
result = bboptimize(
cost_multi,
Method = :borg_moea,
FitnessScheme = ParetoFitnessScheme{2}(is_minimizing = true),
SearchRange = [
(0.1, 1.0), # β_und
(3.0, 14.0), # detection_time
(0.01, 0.1), # death_rate
],
NumDimensions = 3,
MaxTime = 120, # 2 минуты
TraceMode = :compact,
)
best = best_candidate(result)
fitness = best_fitness(result)
println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")
println("Достигнутые показатели:")
println("Пик заболеваемости: $(fitness[1])")
println("Доля умерших: $(fitness[2])")
save(datadir("optimization_result.jld2"), Dict("best" => best, "fitness" => fitness))

LoadError: UndefVarError: `cost_multi` not defined in `Main`
Suggestion: check for spelling errors or missing imports.